<img src="../images/cads-logo.png" style="height: 100px;" align=left> 
<img src="../images/sklearn-logo.png" style="height: 100px;" align=right>

# Supervised Machine Learning

# Table of Contents

- [Thinking about Model Validation](#Thinking-about-Model-Validation)
- [Cross Validation](#Cross-Validation)
- [Model validation the wrong way](#Model-validation-the-wrong-way)
    - [Question: Can you guess the result of the following cell?](#Question:-Can-you-guess-the-result-of-the-following-cell?)
- [Model validation the right way: Holdout sets](#Model-validation-the-right-way:-Holdout-sets)
- [Model validation via cross-validation](#Model-validation-via-cross-validation)
- [Grid Search](#Grid-Search)

# Thinking about Model Validation

In principle, model validation is very simple: after choosing a model and its hyperparameters, we can estimate how effective it is by applying it to some of the training data and comparing the prediction to the known value.

The following sections first show a naive approach to model validation and why it
fails, before exploring the use of holdout sets and cross-validation for more robust
model evaluation.

# Cross Validation

## Model validation the wrong way

Let's demonstrate the naive approach to validation using the Iris data, which we saw in the previous section. We will start by loading the data:

In [35]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()

In [36]:
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data
y = iris.target

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Shape of X: (150, 4)
Shape of y: (150,)


Next we choose a model and hyperparameters. Here we'll use a *k*-neighbors classifier with ``n_neighbors=1``.
This is a very simple and intuitive model that says "the label of an unknown point is the same as the label of its closest training point:"

<img src='../images/KNN.png'>

In [37]:
from sklearn.neighbors import KNeighborsClassifier
model= KNeighborsClassifier(n_neighbors=1)

Then we train the model, and use it to predict labels for data we already know:

In [38]:
model.fit(X, y)
y_model = model.predict(X)

Finally, we compute the fraction of correctly labeled points:

### Question: Can you guess the result of the following cell?

In [39]:
from sklearn.metrics import accuracy_score
accuracy_score(y, y_model)

1.0

We see an accuracy score of 1.0, which indicates that 100% of points were correctly labeled by our model!
But is this truly measuring the expected accuracy? Have we really come upon a model that we expect to be correct 100% of the time?

As you may have gathered, the answer is no.
In fact, this approach contains a fundamental flaw: *it trains and evaluates the model on the same data*.
Furthermore, the nearest neighbor model is an *instance-based* estimator that simply stores the training data, and predicts labels by comparing new data to these stored points: except in contrived cases, it will get 100% accuracy *every time!*

## Model validation the right way: Holdout sets

So what can be done?
A better sense of a model's performance can be found using what's known as a *holdout set*: that is, we hold back some subset of the data from the training of the model, and then use this holdout set to check the model performance.
This splitting can be done using the ``train_test_split`` utility in Scikit-Learn:

In [40]:
print(X.shape)
print(y.shape)

(150, 4)
(150,)


In [41]:
from sklearn.model_selection import train_test_split

# split the data with 50% in each set
X_train, X_test, y_train, y_test = train_test_split(X, y,random_state=0,train_size=0.5)

#fit the model
model.fit(X_train, y_train)

# fit and evaluate the model on the second set of data
y_model = model.predict(X_test)

accuracy_score(y_test, y_model)

0.9066666666666666

In [42]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(75, 4)
(75,)
(75, 4)
(75,)


We see here a more reasonable result: the nearest-neighbor classifier is about 90% accurate on this hold-out set.
The hold-out set is similar to unknown data, because the model has not "seen" it before.

## Model validation via cross-validation

One disadvantage of using a holdout set for model validation is that we have lost a portion of our data to the model training.
In the above case, half the dataset does not contribute to the training of the model!
This is not optimal, and can cause problems – especially if the initial set of training data is small.

One way to address this is to use *cross-validation*; that is, to do a sequence of fits where each subset of the data is used both as a training set and as a validation set. Visually, it might look something like this:
<img src= "../images/2-fold-CV.png" style="height: 500px;">


Here we do two validation trials, alternately using each half of the data as a holdout set.
Using the split data from before, we could implement it like this:

**Question:**
Write the code that implements the accuracy described on the previous image

In [43]:
# solution
y1_model = model.fit(X_train, y_train).predict(X_test)
y2_model = model.fit(X_test, y_test).predict(X_train)
accuracy_score(y_test, y1_model), accuracy_score(y_train, y2_model)

(0.9066666666666666, 0.96)

What comes out are two accuracy scores, which we could combine (by, say, taking the mean) to get a better measure of the global model performance.
This particular form of cross-validation is a *two-fold cross-validation*—that is, one in which we have split the data into two sets and used each in turn as a validation set.

We could expand on this idea to use even more trials, and more folds in the data—for example, here is a visual depiction of five-fold cross-validation:

<img src='../images/CV.png'/>

Here we split the data into five groups, and use each of them in turn to evaluate the model fit on the other 4/5 of the data.
This would be rather tedious to do by hand, and so we can use Scikit-Learn's ``cross_val_score`` convenience routine to do it succinctly:

In [44]:
from sklearn.model_selection import cross_val_score

model= KNeighborsClassifier(n_neighbors=3)
scores = cross_val_score(model, X, y, cv=2)  
scores

array([0.96, 0.92])

In [45]:
print("Accuracy: {}".format(scores.mean()))

Accuracy: 0.94


By default, the score computed at each cv iteration is the `score` method of the estimator. It is possible to change this by using the scoring parameter. Take a look at all possible values for [scoring parameter](https://scikit-learn.org/stable/modules/model_evaluation.html).


In [46]:
scores_f1 = cross_val_score(model, X, y, cv=2, scoring='f1_macro')   
scores_f1, np.mean(scores_f1)

(array([0.95998399, 0.91987179]), np.float64(0.9399278942346169))

Repeating the validation across different subsets of the data gives us an even better idea of the performance of the algorithm.

Scikit-Learn implements a number of useful cross-validation schemes that are useful in particular situations; these are implemented via iterators in the ``cross_validation`` module.
For example, we might wish to go to the extreme case in which our number of folds is equal to the number of data points: that is, we train on all points but one in each trial.
This type of cross-validation is known as *leave-one-out* cross validation, and can be used as follows:

In [47]:
from sklearn.model_selection import LeaveOneOut

model= KNeighborsClassifier(n_neighbors=3)
scores = cross_val_score(model, X, y, cv=LeaveOneOut())
scores

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

Because we have 150 samples, the leave one out cross-validation yields scores for 150 trials, and the score indicates either successful (1.0) or unsuccessful (0.0) prediction.
Taking the mean of these gives an estimate of the error rate:

In [48]:
scores.mean()

np.float64(0.96)

Other cross-validation schemes can be used similarly.
For a description of what is available in Scikit-Learn, use IPython to explore the ``sklearn.cross_validation`` submodule, or take a look at Scikit-Learn's online [cross-validation documentation](http://scikit-learn.org/stable/modules/cross_validation.html).

**Exercise:**
Try to classify Iris data using KNN for n_neighbors=4. Use 5-fold cross validation and use Accuracy, Precision, Recall, F1-score as evaluation metrics.

In [49]:
# 2. Instantiate the KNN classifier with n_neighbors=4
knn = KNeighborsClassifier(n_neighbors=4)

# 3. Perform 5-fold cross-validation for each evaluation metric
accuracy = cross_val_score(knn, X, y, cv=5, scoring='accuracy')
precision = cross_val_score(knn, X, y, cv=5, scoring='precision_macro')
recall = cross_val_score(knn, X, y, cv=5, scoring='recall_macro')
f1 = cross_val_score(knn, X, y, cv=5, scoring='f1_macro')

# 4. Print the mean scores across the 5 folds
print(f"Accuracy:  {accuracy.mean():.4f}")
print(f"Precision: {precision.mean():.4f}")
print(f"Recall:    {recall.mean():.4f}")
print(f"F1-score:  {f1.mean():.4f}")

Accuracy:  0.9733
Precision: 0.9758
Recall:    0.9733
F1-score:  0.9733


### Exercise
Try to classify 'indian_liver_patient.csv' data using KNN for n_neighbors=5 and Logistic Regression. Use 3-fold cross validation.

This data set contains 416 liver patient records and 167 non liver patient records collected from North East of Andhra Pradesh, India. The "Dataset" column is a class label used to divide groups into liver patient (liver disease) or not (no disease). 

In [50]:
liver = pd.read_csv("../Data/indian_liver_patient.csv")
liver.head(2)

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,0
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,0


In [51]:
liver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 583 entries, 0 to 582
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         583 non-null    int64  
 1   Gender                      583 non-null    object 
 2   Total_Bilirubin             583 non-null    float64
 3   Direct_Bilirubin            583 non-null    float64
 4   Alkaline_Phosphotase        583 non-null    int64  
 5   Alamine_Aminotransferase    583 non-null    int64  
 6   Aspartate_Aminotransferase  583 non-null    int64  
 7   Total_Protiens              583 non-null    float64
 8   Albumin                     583 non-null    float64
 9   Albumin_and_Globulin_Ratio  583 non-null    float64
 10  Dataset                     583 non-null    int64  
dtypes: float64(5), int64(5), object(1)
memory usage: 50.2+ KB


In [52]:
#Assuming 0=no disease and 1=disease
liver.Dataset.value_counts()

Dataset
0    416
1    167
Name: count, dtype: int64

In [53]:
X = liver.drop('Dataset', axis=1)
y = liver.Dataset

In [54]:
X_dum=pd.get_dummies(X, drop_first=True)
X_dum

,Age,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Gender_Male
0,65,0.7,0.1,187,16,18,6.8,3.3,0.90,False
1,62,10.9,5.5,699,64,100,7.5,3.2,0.74,True
2,62,7.3,4.1,490,60,68,7.0,3.3,0.89,True
3,58,1.0,0.4,182,14,20,6.8,3.4,1.00,True
4,72,3.9,2.0,195,27,59,7.3,2.4,0.40,True
...,...,...,...,...,...,...,...,...,...,...
578,60,0.5,0.1,500,20,34,5.9,1.6,0.37,True
579,40,0.6,0.1,98,35,31,6.0,3.2,1.10,True
580,52,0.8,0.2,245,48,49,6.4,3.2,1.00,True
581,31,1.3,0.5,184,29,32,6.8,3.4,1.00,True


In [55]:
### KNN

from sklearn.preprocessing import MinMaxScaler
x_scaled = MinMaxScaler().fit_transform(X_dum)


knn_5= KNeighborsClassifier(n_neighbors=5)

A_scores=cross_val_score(knn_5, x_scaled, y, cv=3, scoring='precision_macro')
A_mean=np.mean(A_scores)
print('Mean of precision: {}'.format(A_mean))


# 3. Perform 5-fold cross-validation for each evaluation metric
precision = cross_val_score(knn_5, x_scaled, y, cv=3, scoring='precision_macro')
recall = cross_val_score(knn_5, x_scaled, y, cv=3, scoring='recall_macro')
f1 = cross_val_score(knn_5, x_scaled, y, cv=3, scoring='f1_macro')

# 4. Print the mean scores across the 5 folds
print(f"Precision: {precision.mean():.4f}")
print(f"Recall:    {recall.mean():.4f}")
print(f"F1-score:  {f1.mean():.4f}")

Mean of precision: 0.5797898665311592
Precision: 0.5798
Recall:    0.5727
F1-score:  0.5674


In [56]:
### Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_5= LogisticRegression()

A_scores=cross_val_score(lr_5, x_scaled, y, cv=3, scoring='precision_macro')
A_mean=np.mean(A_scores)
print('Mean of precision: {}'.format(A_mean))

# 3. Perform 5-fold cross-validation for each evaluation metric
precision = cross_val_score(lr_5, x_scaled, y, cv=3, scoring='precision_macro')
recall = cross_val_score(lr_5, x_scaled, y, cv=3, scoring='recall_macro')
f1 = cross_val_score(lr_5, x_scaled, y, cv=3, scoring='f1_macro')

# 4. Print the mean scores across the 5 folds
print(f"Precision: {precision.mean():.4f}")
print(f"Recall:    {recall.mean():.4f}")
print(f"F1-score:  {f1.mean():.4f}")

Mean of precision: 0.7758208345692875
Precision: 0.7758
Recall:    0.5114
F1-score:  0.4448


# Grid Search
Grid search is the process of performing hyper parameter tuning in order to determine the optimal values for a given model. Scikit-Learn provides automated tools to do this in the grid search module.

Here is an example of using grid search to find the optimal KNN model. This can be set up using Scikit-Learn's ``GridSearchCV`` meta-estimator:

Now let's load breast_cancer data that is available from sklearn datasets.

In [57]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

X = cancer.data
y= cancer.target
X.shape

(569, 30)

In [58]:
np.sum(y)

np.int64(357)

In [59]:
X_train, X_test, y_train, y_test=train_test_split(X,y,random_state=42, train_size=.80,stratify=y)

In [60]:
from sklearn.model_selection import GridSearchCV

param_grid = {'n_neighbors': np.arange(1, 20),
              'p': [1,2],
              'weights': ['uniform','distance']}

grid = GridSearchCV(KNeighborsClassifier(), 
                    param_grid, scoring='precision_macro',
                    cv=3)

In [61]:
# What is the total number of models that grid search builds


Notice that like a normal estimator, this has not yet been applied to any data.
Calling the ``fit()`` method will fit the model at each grid point, keeping track of the scores along the way:

In [62]:
grid.fit(X_train, y_train)

,estimator,KNeighborsClassifier()
,param_grid,"{'n_neighbors': array([ 1, 2... 18, 19]), 'p': [1, 2], 'weights': ['uniform', 'distance']}"
,scoring,'precision_macro'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,np.int64(7)


Now that this is fit, we can ask for the best parameters as follows:

In [63]:
grid.best_params_

{'n_neighbors': np.int64(7), 'p': 1, 'weights': 'uniform'}

In [64]:
grid

,estimator,KNeighborsClassifier()
,param_grid,"{'n_neighbors': array([ 1, 2... 18, 19]), 'p': [1, 2], 'weights': ['uniform', 'distance']}"
,scoring,'precision_macro'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,np.int64(7)


Finally, if we wish, we can use the best model and show the fit to our data using code from before:

In [65]:
model = grid.best_estimator_
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
print('classification_report:\n',classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

classification_report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.93        42
           1       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



array([[38,  4],
       [ 2, 70]])

The grid search provides many more options, including the ability to specify a custom scoring function, to parallelize the computations, to do randomized searches, and more.

## Exercise:

Load the cancer dataset and choose the best classification algorithm with the best hyperparameters.

- Define X and y

- To simplify, remove missing values

- Split data to train and test

- Use 5 fold cross validation and grid search on train data

- Choose appropriate validation metric

- Set grid parameters for each classification algorithm

- Build the best models for each classification algorithm according to the best estimator (best hyperparameters) given by the grid search

- Compare the performance of the algorithms with the best hyperparametrs on the test data according to confusion matrix, recall, precision, F1, and auc metrics

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, MinMaxScaler
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
df=pd.read_csv('../data/breast_cancer_wisconsin.csv')
df


,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
694,776715,3,1,1,1,3,2.0,1,1,1,0
695,841769,2,1,1,1,2,1.0,1,1,1,0
696,888820,5,10,10,3,7,3.0,8,10,2,1
697,897471,4,8,6,4,3,4.0,10,6,1,1


In [5]:
from sklearn.neighbors import KNeighborsClassifier

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Id               699 non-null    int64  
 1   Cl.thickness     699 non-null    int64  
 2   Cell.size        699 non-null    int64  
 3   Cell.shape       699 non-null    int64  
 4   Marg.adhesion    699 non-null    int64  
 5   Epith.c.size     699 non-null    int64  
 6   Bare.nuclei      683 non-null    float64
 7   Bl.cromatin      699 non-null    int64  
 8   Normal.nucleoli  699 non-null    int64  
 9   Mitoses          699 non-null    int64  
 10  Class            699 non-null    int64  
dtypes: float64(1), int64(10)
memory usage: 60.2 KB


In [6]:
# Clean up unnecessary ID or unnamed columns if present
if 'id' in df.columns:
    df = df.drop(columns=['id'])

df = df.dropna()  # Remove rows with missing values


# ---------------------------------------------------------
# Step 2: Remove missing values
# ---------------------------------------------------------

if 'Class' in df.columns:
    y = df['Class']
    X = df.drop(columns=['Class'])
else:
    y = df['target']
    X = df.drop(columns=['target'])

# ---------------------------------------------------------
# Step 3: Split data into Train and Test sets
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Step 4 & 5: Validation Metric & Parameter Grids Setup
# Metric Choice: 'f1' balances false positives and false negatives, 
# which is critical in medical diagnostic classification tasks.
# ---------------------------------------------------------
scoring_metric = 'f1'

classifiers = {
    'KNN': {
        'estimator': Pipeline([
            ('scaler', MinMaxScaler()),
            ('clf', KNeighborsClassifier())
        ]),
        'params': {
            'clf__n_neighbors': [3, 5, 7, 9, 11],
            'clf__weights': ['uniform', 'distance'],
            'clf__metric': ['euclidean', 'manhattan']
        }
    },
    'Decision Tree': {
        'estimator': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [3, 5, 10, None],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    },
    'Logistic Regression': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=10000, random_state=42))
        ]),
        'params': {
            'clf__C': [0.01, 0.1, 1, 10, 100],
            'clf__solver': ['lbfgs', 'liblinear']
        }
    },
    'SVM': {
        'estimator': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(probability=True, random_state=42))
        ]),
        'params': {
            'clf__C': [0.1, 1, 10, 100],
            'clf__kernel': ['linear', 'rbf'],
            'clf__gamma': ['scale', 'auto']
        }
    },
    'Polynomial Logistic Regression': {
        'estimator': Pipeline([
            ('poly', PolynomialFeatures(degree=2)),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=10000, random_state=42))
        ]),
        'params': {
            'clf__C': [0.01, 0.1, 1, 10]
        }
    }
}

# ---------------------------------------------------------
# Step 6: 5-Fold Cross Validation & Grid Search on Train Data
# ---------------------------------------------------------
best_models = {}

print("=== Grid Search Results (5-Fold CV) ===")
for name, clf_info in classifiers.items():
    grid = GridSearchCV(
        estimator=clf_info['estimator'],
        param_grid=clf_info['params'],
        cv=5,
        scoring=scoring_metric,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    
    print(f"\nClassifier: {name}")
    print(f"  Best Parameters: {grid.best_params_}")
    print(f"  Best 5-Fold CV F1-Score: {grid.best_score_:.4f}")

# ---------------------------------------------------------
# Step 7: Compare Performance of Best Estimators on Test Data
# ---------------------------------------------------------
results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    
    # Retrieve class probabilities for AUC score calculation
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = model.decision_function(X_test)
        
    cm = confusion_matrix(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Algorithm': name,
        'Confusion Matrix [[TN, FP], [FN, TP]]': cm.tolist(),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC': round(auc, 4)
    })

comparison_df = pd.DataFrame(results)

print("\n=== Model Evaluation on Test Data ===")
print(comparison_df.to_string(index=False))

=== Grid Search Results (5-Fold CV) ===

Classifier: KNN
  Best Parameters: {'clf__metric': 'euclidean', 'clf__n_neighbors': 7, 'clf__weights': 'uniform'}
  Best 5-Fold CV F1-Score: 0.9632

Classifier: Decision Tree
  Best Parameters: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 2}
  Best 5-Fold CV F1-Score: 0.9422

Classifier: Logistic Regression
  Best Parameters: {'clf__C': 0.01, 'clf__solver': 'liblinear'}
  Best 5-Fold CV F1-Score: 0.9584

Classifier: SVM
  Best Parameters: {'clf__C': 0.1, 'clf__gamma': 'scale', 'clf__kernel': 'linear'}
  Best 5-Fold CV F1-Score: 0.9557

Classifier: Polynomial Logistic Regression
  Best Parameters: {'clf__C': 0.1}
  Best 5-Fold CV F1-Score: 0.9576

=== Model Evaluation on Test Data ===
                     Algorithm Confusion Matrix [[TN, FP], [FN, TP]]  Precision  Recall  F1-Score    AUC
                           KNN                    [[85, 4], [2, 46]]     0.9200  0.9583    0.9388 0.9831
                 Decision Tree         

In [ ]:
StandardScaler
=== Model Evaluation on Test Data ===
                     Algorithm Confusion Matrix [[TN, FP], [FN, TP]]  Precision  Recall  F1-Score    AUC
                           KNN                    [[85, 4], [2, 46]]     0.9200  0.9583    0.9388 0.9754
                 Decision Tree                    [[85, 4], [1, 47]]     0.9216  0.9792    0.9495 0.9771
           Logistic Regression                    [[85, 4], [1, 47]]     0.9216  0.9792    0.9495 0.9934
                           SVM                    [[85, 4], [2, 46]]     0.9200  0.9583    0.9388 0.9932
Polynomial Logistic Regression                    [[85, 4], [1, 47]]     0.9216  0.9792    0.9495 0.9923

**Knn**

**Decision Tree**

**Logistic Regression**

**Support Vector Machine (SVM)**

**PipeLine: Polynomial Logistic Regression**

**Compare the best models on the test data**